In [ ]:
pip install findspark

In [1]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


Successfully connected to HYPE on port 1521.


In [1]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [2]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
#mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-11.2.4.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [3]:
# SQL Server connection properties
# cannot make partition with char column
from pyspark.sql import functions as sf
import time
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=AdventureSales;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
# --- Configuration (reuse previous variables) ---
#sql_server_url = "your_sql_server_url"
#sqlserver_url = "your_sqlserver_url"
# ... credentials ...
source_view_name = "sttwk1b" # This view has YEAR_WEEK column for partitioning
sqlsvr_destination_table = "STTWK_2225"   # This table does NOT have YEAR_WEEK

partition_column_name = "year_week_int"
# Assume year_week_list has 181 specific values:
# year_week_list = ['202201', '202202', ... ]
year_week_list = [str(y) + str(w).zfill(2) for y in range(2022, 2022) for w in range(1, 53)] # This generates too many, but illustrates the format
# Create the specific predicates list
predicates = [f"{partition_column_name} = '{yw_value}'" for yw_value in year_week_list]


# 2. Read data from SQL Server using the 181 predicates
#    We define the columns explicitly to IGNORE the year_week column
#    that is only used for the WHERE clause (predicates)
#columns_to_select = "branch_id, sp_id, custd_id, prod_id, sls_id, yr, mth, wk, net_qty, net_amt, grs_qty, grs_amt, rtn_qty, rtn_amt"
partition_column_name = "year_week_int"
# Start the timer
start_time = time.time()

# Parameters for parallel read (adjust as needed for your table and data)
# The partitionColumn must be a numeric, date, or timestamp column.
#partition_column = "week" 
#lower_bound = 1
#upper_bound = 13 # Max ID in your table, e.g.
#num_read_partitions = 13 # Read in 10 partitions
    #.option("url", sqlserver_jdbc_url) \
    #.option("partitionColumn", partition_column) \
    #.option("lowerBound", lower_bound) \
    #.option("upperBound", upper_bound) \
    #.option("numPartitions", num_read_partitions) \
    #.option("customSql", f"SELECT {columns_to_select} FROM {source_view_name}") \
df_partitioned = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", source_view_name) \
    .option("fetchsize", 1000) \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .load(predicates=predicates) 
# Note: When using predicates with custom selects, ensure the source table/view in dbtable 
# is correct and the predicates column is available in the source (which it is).
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions.")
df_partitioned.printSchema() # Verify year_week is not in the schema
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")
# Optional: Check the number of partitions created
#print(f"DataFrame has {sql_server_df.rdd.getNumPartitions()} partitions after read.")


DataFrame has 1 partitions.
root
 |-- branch_id: string (nullable = true)
 |-- sp_id: string (nullable = true)
 |-- custd_id: string (nullable = true)
 |-- prod_id: string (nullable = true)
 |-- sls_id: string (nullable = true)
 |-- year: decimal(4,0) (nullable = true)
 |-- month: string (nullable = true)
 |-- week: decimal(2,0) (nullable = true)
 |-- net_qty: decimal(12,4) (nullable = true)
 |-- net_amt: decimal(18,4) (nullable = true)
 |-- grs_qty: decimal(12,4) (nullable = true)
 |-- grs_amt: decimal(18,4) (nullable = true)
 |-- rtn_qty: decimal(10,2) (nullable = true)
 |-- rtn_amt: decimal(15,2) (nullable = true)
 |-- year_week_int: decimal(9,0) (nullable = true)

Total time taken for transfer: 2.57 seconds


In [4]:
from pyspark.sql.functions import countDistinct, col

# Start the timer
start_time = time.time()

# Option A: Repartition by the character column (hash partitioning)
# The data associated with each unique value in 'char_column' will go to a single partition.
#df_to_write = sql_server_df.repartition(181, "year_week_int")
num_read_partitions = df_partitioned.agg(countDistinct(col("year_week_int"))).collect()[0][0]
print(f"Detected {num_read_partitions} unique year_week_int values.")
df_to_write = df_partitioned.repartition(num_read_partitions, "year_week_int")
# Option B: Repartition generally (even hash distribution)
# Use this if you don't care about co-locating data by the char column, 
# or if the data in 'your_char_column_name' is very skewed.
# df_to_write = sql_server_df.repartition(10) 

print(f"DataFrame has {df_to_write.rdd.getNumPartitions()} partitions before write.")
predicates = [f"{partition_column_name} = '{yw_value}'" for yw_value in year_week_list]
#columns_to_select = "branch_id, sp_id, custd_id, prod_id, sls_id, yr, mth, wk, net_qty, net_amt, grs_qty, grs_amt, rtn_qty, rtn_amt"

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

Detected 181 unique year_week_int values.
DataFrame has 181 partitions before write.
Total time taken for transfer: 367.60 seconds


In [21]:
# Drop the temporary 'year_week_int' column before writing to Oracle
# just do this cell if dont want to include them !!!!
#df_to_write = df_partitioned.drop("year_week_int")


In [5]:
# Assuming you have successfully run the previous code block which creates `df_partitioned`

import time

# --- Configuration for Destination Table ---
sqlsvr_destination_table = "sttwk_2225" 

# Define write properties with bulkCopy AND truncate enabled
write_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "encrypt": "true",
    "trustServerCertificate": "true",
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}

print(f"Attempting to truncate and write to destination table: {sqlsvr_destination_table}")
#print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions for writing.")

# Start timer for write operation
write_start_time = time.time()

try:
    df_to_write.write \
        .mode("overwrite") \
        .jdbc(
            url=sql_server_url, 
            table=sqlsvr_destination_table, 
            properties=write_properties
        )
    
    write_end_time = time.time()
    write_total_time = write_end_time - write_start_time

    print("-" * 50)
    print(f"Data successfully truncated and inserted into table {sqlsvr_destination_table}.")
    print(f"Write operation completed in: {write_total_time:.2f} seconds")
    print("-" * 50)

except Exception as e:
    print(f"An error occurred during the write operation: {e}")   

Attempting to truncate and write to destination table: sttwk_2225
--------------------------------------------------
Data successfully truncated and inserted into table sttwk_2225.
Write operation completed in: 1094.40 seconds
--------------------------------------------------


In [ ]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
#TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
TARGET_TABLE = "sttwk_2225"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}
# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    *
FROM sttwk1b 
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")  

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df.repartition(52)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")         